In [68]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    count, sum as _sum, avg, round as _round, 
    min as _min, max as _max, 
    to_timestamp, col, window, desc
)

# ==========================================
# Inicjalizacja i wczytanie danych
# ==========================================
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy") 

Spark 4.0.0-preview2 — gotowy


In [69]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema() 
df.show(10, truncate=False) 

# konwersja timestamp
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywno

In [70]:
# ==========================================
# Zadanie 2.1 — Liczba transakcji i suma przychodów per sklep
# ==========================================
print("\n--- Zadanie 2.1 ---")
store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()


--- Zadanie 2.1 ---
+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [71]:
# ==========================================
# Zadanie 2.2 — Statystyki per kategoria
# ==========================================
print("\n--- Zadanie 2.2 ---")
df.groupBy("category").agg(
    _round(_sum("amount"), 2).alias("suma_PLN"),
    _round(_min("amount"), 2).alias("min_PLN"),
    _round(_max("amount"), 2).alias("max_PLN")
).orderBy("category").show()


--- Zadanie 2.2 ---
+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [72]:
# ==========================================
# Zadanie 3.1 — Liczba transakcji per godzina (tumbling 1h)
# ==========================================
print("\n--- Zadanie 3.1 ---")
hourly = (
    df.groupBy(window("timestamp", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)

# wyciągnięcie pól start i end, żeby ładniej wyświetlić
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)


--- Zadanie 3.1 ---
+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|1112     |411159.81|
|2026-04-12 08:30:00|2026-04-12 09:00:00|2038     |830751.49|
|2026-04-12 09:00:00|2026-04-12 09:30:00|2405     |922282.11|
|2026-04-12 09:30:00|2026-04-12 10:00:00|2256     |973948.1 |
|2026-04-12 10:00:00|2026-04-12 10:30:00|1440     |583693.29|
|2026-04-12 10:30:00|2026-04-12 11:00:00|749      |289709.95|
+-------------------+-------------------+---------+---------+



In [73]:
# ==========================================
# Zadanie 3.2 — Okna 30-minutowe per sklep
# ==========================================
print("\n--- Zadanie 3.2 ---")
df.groupBy(window("timestamp", "30 minutes"), "store").agg(
    count("tx_id").alias("liczba_tx"),
    _round(_sum("amount"), 2).alias("suma_PLN")
).orderBy("window", "store").show(truncate=False)


--- Zadanie 3.2 ---
+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:0

In [74]:
# ==========================================
# Zadanie 3.3 — W której godzinie sklep "Kraków" miał najwyższy przychód?
# ==========================================
print("\n--- Zadanie 3.3 ---")
df.filter(col("store") == "Kraków") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(_round(_sum("amount"), 2).alias("suma_PLN")) \
  .orderBy(desc("suma_PLN")) \
  .show(1, truncate=False)


--- Zadanie 3.3 ---
+------------------------------------------+---------+
|window                                    |suma_PLN |
+------------------------------------------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86|
+------------------------------------------+---------+
only showing top 1 row



In [75]:
# ==========================================
# Zadanie 4.1 — Okno 1h, krok 30 minut (sliding)
# ==========================================
print("\n--- Zadanie 4.1 ---")
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)


--- Zadanie 4.1 ---
+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [76]:
# ==========================================
# Zadanie 4.2 — Porównaj tumbling vs sliding
# ==========================================
print("\n--- Zadanie 4.2 ---")
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h):        {tumbling_rows} okien")
print(f"Sliding  (1h / 30min): {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# ODPOWIEDŹ: Okno przesuwne (sliding) tworzy nakładające się na siebie interwały (np. co pół godziny powstaje nowe okno obejmujące pełną godzinę). 
# Kluczowy jest fakt, że okna się na siebie nakładają, przez co jedna transakcja może wpaść do kilku okien jednocześnie.


--- Zadanie 4.2 ---
Tumbling (1h):        3 okien
Sliding  (1h / 30min): 7 okien


In [77]:
# ==========================================
# Część 5: Pytania kontrolne
# ==========================================
# 1. Ile transakcji jest w oknie 09:00–10:00?
# ODPOWIEDŹ: 4661
#
# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
# ODPOWIEDŹ: Pierwsze grupuje wszystkie rekordy w historii wyłącznie po nazwie sklepu (jeden wynik na sklep). Drugie dzieli historię na przedziały czasowe i grupuje po sklepie osobno DLA KAŻDEGO przedziału.
#
# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
# ODPOWIEDŹ: Dwa okna. Jedno zaczynające się o 09:00 (09:00-10:00), a drugie zaczynające się o 09:30 (09:30-10:30), wynika to z faktu, że przedziały są domyślnie lewostronnie domknięte i prawostronnie otwarte. 

In [78]:
# ==========================================
# Praca domowa
# ==========================================
print("\n--- Praca domowa 1: Gdańsk, najniższa średnia kwota ---")
df.filter(col("store") == "Gdańsk") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(_round(avg("amount"), 2).alias("srednia_PLN")) \
  .orderBy("srednia_PLN") \
  .show(1, truncate=False)

print("\n--- Praca domowa 2: Ile transakcji per kategoria w oknie 09:00-09:30 ---")
# Grupowanie w 30-minutowe okna i od razu filtrowanie tego konkretnego
df.groupBy(window("timestamp", "30 minutes"), "category") \
  .agg(count("tx_id").alias("liczba_tx")) \
  .filter(col("window.start").cast("string").like("%09:00:00")) \
  .orderBy("category") \
  .show(truncate=False)

print("\n--- Praca domowa 3: W której ćwierć godzinie był szczyt transakcji ---")
df.groupBy(window("timestamp", "15 minutes")) \
  .agg(count("tx_id").alias("liczba_tx")) \
  .orderBy(desc("liczba_tx")) \
  .show(1, truncate=False)

# Zakończenie sesji Spark
spark.stop()


--- Praca domowa 1: Gdańsk, najniższa średnia kwota ---
+------------------------------------------+-----------+
|window                                    |srednia_PLN|
+------------------------------------------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01     |
+------------------------------------------+-----------+
only showing top 1 row


--- Praca domowa 2: Ile transakcji per kategoria w oknie 09:00-09:30 ---
+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+


--- Praca domowa 3: W 